# EV Adoption Behavior - 01. Data Understanding

**Business problem:** An EV company wants to understand what factors influence a customer's likelihood of adopting an EV, and present actionable insights via an interactive dashboard.

**Goal of this notebook:** get a first honest look at the data - shape, types, missingness, target balance, and anything that looks broken - before deciding how to clean it.

## 1. Load & basic shape

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('global_ev_adoption_behavior_2026.csv')
print(df.shape)
df.head()

(50000, 23)


,age,annual_income,education_level,city_type,daily_commute_km,weekly_travel_distance_km,current_vehicle_type,vehicle_age_years,fuel_expense_per_month,charging_station_accessibility,...,environmental_awareness_score,government_incentive_awareness,technology_affinity_score,range_anxiety_score,battery_replacement_concern,ev_knowledge_score,previous_ev_experience,ev_adoption_likelihood,monthly_energy_consumption_kwh,monthly_charging_cost
0,56,23019.0,High School,Suburban,39.8,205.7,Hatchback,6.1,317.1,3.9,...,8.3,6.4,5.2,4.6,3.5,6.9,0,High,199.8,28.5
1,46,26440.0,High School,Suburban,34.6,218.4,Sedan,4.4,290.0,4.5,...,7.8,10.0,5.7,5.5,7.4,6.9,1,High,139.8,22.9
2,46,57167.0,PhD,Suburban,30.5,177.7,Sedan,0.4,201.4,6.3,...,7.6,6.4,7.8,5.3,6.2,6.8,0,High,158.0,38.5
3,23,15841.0,Master,Suburban,44.6,325.9,SUV,0.0,407.3,5.2,...,6.2,5.6,5.1,7.9,5.7,7.2,0,Low,207.0,45.0
4,50,51571.0,Master,Urban,52.4,281.0,SUV,5.2,458.4,4.3,...,9.7,9.9,8.2,2.6,4.7,8.1,0,High,195.6,43.9


In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Missing values

Check overall null counts, then narrow down to just the affected columns and check whether the same rows are missing multiple fields at once (systemic) or missingness is scattered (incidental).

In [ ]:
missing_counts = df.isnull().sum()
missing_counts[missing_counts > 0]

In [ ]:
missing_cols = missing_counts[missing_counts > 0].index.tolist()
missing_cols

In [ ]:
# overlap check: how many rows are missing 0, 1, 2, 3 of these fields at once?
df[missing_cols].isnull().sum(axis=1).value_counts()

**Finding:** missingness is scattered, not systemic - the vast majority of incomplete rows are missing only *one* of the three fields (`education_level`, `charging_station_accessibility`, `ev_knowledge_score`), not all at once. That points toward imputation rather than dropping rows.

## 3. Target variable - `ev_adoption_likelihood`

Check class balance. A skewed target changes how later charts should be read - a segment showing "high adoption" might just reflect the overall skew, not a genuinely EV-friendly segment.

In [3]:
df['ev_adoption_likelihood'].value_counts(normalize=True) * 100

ev_adoption_likelihood
High      59.340
Medium    24.156
Low       16.504
Name: proportion, dtype: float64

## 4. Data quality checks on suspicious columns

In [ ]:
# vehicle_age_years == 0 -- data error, or legitimately a brand-new vehicle?
zero_age = df[df['vehicle_age_years'] == 0]
print(len(zero_age))
zero_age.head(10)

In [4]:
# negative fuel_expense_per_month -- can't be a real value
neg_fuel = df[df['fuel_expense_per_month'] < 0]
print(len(neg_fuel))
neg_fuel['fuel_expense_per_month'].describe()

271


count    271.000000
mean     -19.008118
std       16.709722
min      -99.700000
25%      -27.100000
50%      -15.000000
75%       -5.850000
max       -0.300000
Name: fuel_expense_per_month, dtype: float64

## 5. Distribution shape for the numeric columns with missing values

Mean vs. median vs. skew tells us whether mean-imputation is safe, or whether we need median (or a group-wise median).

In [ ]:
df[['charging_station_accessibility', 'ev_knowledge_score']].agg(['mean', 'median', 'skew'])

In [5]:
df.groupby('city_type')['charging_station_accessibility'].median()

city_type
Rural       5.0
Suburban    5.0
Urban       7.0
Name: charging_station_accessibility, dtype: float64

In [6]:
df.groupby('city_type')['ev_knowledge_score'].median()

city_type
Rural       7.0
Suburban    7.0
Urban       7.0
Name: ev_knowledge_score, dtype: float64

## 6. Categorical columns - check cardinality / spelling consistency

In [7]:
for col in ['education_level', 'city_type', 'current_vehicle_type']:
    print(col, '->', df[col].unique())


education_level -> <ArrowStringArray>
['High School', 'PhD', 'Master', 'Bachelor', nan]
Length: 5, dtype: str
city_type -> <ArrowStringArray>
['Suburban', 'Urban', 'Rural']
Length: 3, dtype: str
current_vehicle_type -> <ArrowStringArray>
['Hatchback', 'Sedan', 'SUV', 'Truck']
Length: 4, dtype: str


## Summary of findings

- Target `ev_adoption_likelihood` is skewed toward **High** - keep this in mind when interpreting later segment comparisons.
- `vehicle_age_years == 0` - need to confirm this is a legitimate "brand new vehicle" case, not a data error, before deciding to keep it.
- `fuel_expense_per_month` has negative values - these are data entry errors and need correcting.
- 3 columns have missing values, ~3% of rows, scattered (not systemic):
  - `education_level` (categorical) → impute with mode
  - `charging_station_accessibility` (numeric, symmetric, but varies meaningfully by `city_type` - Urban ~7 vs. Rural/Suburban ~5) → impute with **median per city_type**
  - `ev_knowledge_score` (numeric, mildly left-skewed, no real difference across `city_type`) → impute with **overall median**

These decisions carry into `02_data_cleaning.ipynb`.